# Ollama-Registries: Modelle laden und verwalten

**Lernziel:** Wir lernen den Weg eines Modells von der **Registry** auf den **lokalen Rechner** kennen und prüfen anschliessend, wie sich ein sehr kleines Modell verwenden lässt.

Die Kernfragen: **Welche Modelle sind bereits vorhanden? Woher kommt ein neues Modell? Was ändert `ollama pull`? Wie kann ich das Modell danach anzeigen und starten?**



## 1. Lokalen Modellbestand anzeigen

`ollama list` zeigt die **heruntergeladenen Modelle** samt Grösse und weiteren Angaben. Die Liste zeigt nicht den gesamten Inhalt einer Registry.

**Beobachtet:** Ist `smollm2:135m` bereits vorhanden? Welche Modelle habt ihr aus den vorherigen Übungen?


In [ ]:
%%bash
ollama list

### Nur falls Ollama nicht erreichbar ist

Wenn `ollama list` einen Verbindungsfehler meldet, könnt ihr den **lokalen Ollama-Dienst neu starten** und die Liste erneut aufrufen. Diese Zelle ist für die Fehlerbehebung gedacht und kann sonst übersprungen werden.

In [ ]:
%%bash
sudo systemctl restart ollama
ollama list

## 2. Ein neues Modell aus der Registry laden

In einer **Modell-Registry** werden Modelle und verschiedene Versionen bzw. Tags bereitgestellt. In der [Ollama-Modellbibliothek](https://ollama.com/library/smollm2) findet ihr zum Beispiel **SmolLM2**.

Mit `ollama pull smollm2:135m` laden wir eine bestimmte Modellvariante in den lokalen Ollama-Modellspeicher. Ein erneuter Aufruf prüft die vorhandenen Daten und lädt nur, was benötigt wird.

**Beobachtet:** Fortschritt des Downloads und Ergebnis. Die Modellgewichte sind danach lokal vorhanden – das Modell beantwortet dadurch aber noch keine Frage.


In [ ]:
%%bash
ollama pull smollm2:135m

## 3. Prüfen, was installiert wurde

Ruft die lokale Liste erneut auf und zeigt danach **Details zum Modell** an.

- `ollama list`: Welche Modelle sind jetzt lokal vorhanden?
- `ollama show`: Welche Informationen gibt Ollama zu einem konkreten Modell aus?

**Vergleicht die Liste vor und nach dem Pull.**


In [ ]:
%%bash
ollama list
ollama show smollm2:135m

## 4. Umgebung einrichten

Wir erstellen anhand der Vorlage `env.py` eine eigene Umgebung. 

Diese dient als zentrale Konfiguration und kann in den nachfolgenden Übungen wiederverwendet werden.

In [ ]:
%%bash
source ~/data/env.py
cat <<EOF | tee ~/data/env-ollama.py
OPENAI_API_KEY="ollama"
HF_TOKEN=""
AI_KUBECONFIG="$AI_KUBECONFIG"
AI_MODEL="llama3.1:8b-instruct-q4_K_M"
AI_NAME="ollama"
AI_IP=""
AI_BASE_URL="http://localhost:11434/v1"
EOF

## 5. Das heruntergeladene Modell verwenden

Erst jetzt führen wir mit `ollama run` eine **Inferenz** aus. Wir stellen SmolLM2 und dem bereits installierten **Llama 3.1** dieselbe einfache Frage. Der Fokus liegt zunächst darauf, dass **beide Modelle lokal gestartet werden können**.

**Beobachtet:** Unterschied zwischen *Modell herunterladen* (`pull`) und *Modell benutzen* (`run`).


In [ ]:
%%bash
source ~/data/env-ollama.py

echo "===== SmolLM2 (135M) ====="
# Entspricht ollama run smollm2:135m

curl -fsS "${AI_BASE_URL}/chat/completions" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "smollm2:135m",
    "messages": [
      {
        "role": "user",
        "content": "Wer war John F. Kennedy? Antworte auf Deutsch."
      }
    ],
    "stream": false
  }' | jq -r '.choices[0].message.content'

echo
echo "===== Llama 3.1 (8B) ====="
# Entspricht ollama run llama3.1:8b-instruct-q4_K_M

curl -fsS "${AI_BASE_URL}/chat/completions" \
  -H "Content-Type: application/json" \
  -d '{
    "model": "llama3.1:8b-instruct-q4_K_M",
    "messages": [
      {
        "role": "user",
        "content": "Wer war John F. Kennedy? Antworte auf Deutsch."
      }
    ],
    "stream": false
  }' | jq -r '.choices[0].message.content'

## 6. Zusatzversuch – wo liegen die Grenzen kleiner Modelle?

**Erst nachdem Download, Modellanzeige und Start funktioniert haben**, vergleichen wir die **Qualität** anhand einer etwas anspruchsvolleren Aufgabe: einen Python-REST-Client vorschlagen.

**Beobachtet:** Ist die Antwort fachlich plausibel? Welche Lücken oder Fehler erkennt ihr?


In [ ]:
%%bash
source ~/data/env-ollama.py
PROMPT="Schreibe ein kurzes Python-Beispiel für einen REST-Client."

for MODEL in smollm2:135m llama3.1:8b-instruct-q4_K_M; do
    echo
    echo "===== $MODEL ====="

    # Entspricht ollama run, jedoch ohne Streaming
    curl -fsS "${AI_BASE_URL}/chat/completions" \
      -H "Content-Type: application/json" \
      -d "$(jq -n \
        --arg model "$MODEL" \
        --arg prompt "$PROMPT" \
        '{
          model: $model,
          messages: [
            {role: "user", content: $prompt}
          ],
          stream: false
        }')" |
      jq -r '.choices[0].message.content'

    echo
done


## Auswertung – Registry, Download und Inferenz unterscheiden

| Begriff / Befehl | Bedeutung |
|---|---|
| **Registry** | Bezugsquelle für veröffentlichte Modelle und Tags |
| **`ollama pull`** | Eine bestimmte Modellversion lokal herunterladen |
| **`ollama list`** | Lokal gespeicherte Modelle anzeigen |
| **`ollama show`** | Informationen zu einem Modell anzeigen |
| **`ollama run`** | Ein lokales Modell verwenden und eine Anfrage beantworten lassen |

**Erkenntnis:** Ein Modell kann aus einer Registry bezogen, lokal verwaltet und später ohne erneuten Download verwendet werden. **Die Modellgrösse beeinflusst Ressourcenbedarf und oft auch die Antwortqualität**, ist aber keine Garantie für korrekte Ergebnisse.

**Transfer:** Sucht in der [Ollama-Modellbibliothek](https://ollama.com/library) eine weitere Modellvariante. Erklärt, welches Kommando sie laden und welches Kommando ihre Installation nachweisen würde. Ein Download ist dafür nicht erforderlich.
